# 01 — Autograd Basics: `.backward()`, `.grad`, and `torch.no_grad()`

**Overview:** verify autograd against `16_backpropagation`'s hand-derived gradients, first on
a tiny scalar example, then on the full 2-layer MLP; demonstrate the `.grad` accumulation
gotcha; use `torch.no_grad()` for inference.

**Required libraries:** `torch`, `numpy`.

Connects to: `18_pytorch_basics` (tensors) and `16_backpropagation` (the exact gradients this
notebook automates). Leads to: `20_pytorch_models`.

### Abbreviations used in this notebook

| Term | Full form | What it is |
|---|---|---|
| **MLP** | Multi-Layer Perceptron | Feed-forward network of fully connected layers with non-linear activations. |
| **CUDA** | Compute Unified Device Architecture | NVIDIA's GPU programming platform; PyTorch's `cuda` device. |
| **CPU** | Central Processing Unit | General-purpose processor; few powerful cores. |
| **MPS** | Metal Performance Shaders | Apple's GPU backend for PyTorch on Apple-silicon Macs. |

## 0. Environment

In [1]:
import torch
import numpy as np
print("torch:", torch.__version__)
torch.manual_seed(42)


torch: 2.14.1


## 1. The tiny scalar example from `16_backpropagation`, via autograd

**Recap:** `a=2, b=3, c=a*b, d=c+1, L=d^2`. We hand-derived `dL/da` and `dL/db` there --
autograd should produce the exact same numbers automatically.

In [2]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

c = a * b
d = c + 1
L = d ** 2

L.backward()   # walks the graph backward, computing dL/da and dL/db automatically

print("autograd dL/da:", a.grad.item())
print("autograd dL/db:", b.grad.item())

# Hand-derived values from 16_backpropagation: dL/da = 2d*b, dL/db = 2d*a
d_value = (a.item() * b.item()) + 1
hand_derived_da = 2 * d_value * b.item()
hand_derived_db = 2 * d_value * a.item()
print("hand-derived dL/da:", hand_derived_da)
print("hand-derived dL/db:", hand_derived_db)

assert abs(a.grad.item() - hand_derived_da) < 1e-5
assert abs(b.grad.item() - hand_derived_db) < 1e-5
print("\nautograd matches the hand-derived chain-rule calculation exactly.")


autograd dL/da: 42.0
autograd dL/db: 28.0
hand-derived dL/da: 42.0
hand-derived dL/db: 28.0

autograd matches the hand-derived chain-rule calculation exactly.


## 2. The 2-layer MLP from `15_deep_learning`/`16_backpropagation`, rebuilt in PyTorch

🎯 **Interview Tip:** this is the direct comparison that should make autograd stop feeling like
a black box -- it's computing the SAME dW1/db1/dW2/db2 as the from-scratch NumPy backward pass.

In [3]:
# Recreate the exact same weights and data as 16_backpropagation's NumPy example, as tensors.
rng = np.random.default_rng(42)
batch_size, in_features, hidden_units, out_classes = 16, 5, 8, 3

X_np = rng.normal(size=(batch_size, in_features))
y_np = rng.integers(0, out_classes, size=batch_size)
W1_np = rng.normal(scale=0.3, size=(in_features, hidden_units))
W2_np = rng.normal(scale=0.3, size=(hidden_units, out_classes))

X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.long)
W1 = torch.tensor(W1_np, dtype=torch.float32, requires_grad=True)
b1 = torch.zeros(hidden_units, requires_grad=True)
W2 = torch.tensor(W2_np, dtype=torch.float32, requires_grad=True)
b2 = torch.zeros(out_classes, requires_grad=True)

# Forward pass -- identical math to the NumPy version, autograd builds the graph as we go.
Z1 = X @ W1 + b1
A1 = torch.relu(Z1)
Z2 = A1 @ W2 + b2
loss = torch.nn.functional.cross_entropy(Z2, y)

print("loss:", loss.item())
loss.backward()

print("\nW1.grad shape:", W1.grad.shape, "== W1 shape:", W1.shape)
print("W2.grad shape:", W2.grad.shape, "== W2 shape:", W2.shape)
assert W1.grad.shape == W1.shape
assert W2.grad.shape == W2.shape
assert not torch.allclose(W1.grad, torch.zeros_like(W1.grad))   # gradients are non-trivial


loss: 1.1216627359390259

W1.grad shape: torch.Size([5, 8]) == W1 shape: torch.Size([5, 8])
W2.grad shape: torch.Size([8, 3]) == W2 shape: torch.Size([8, 3])


## 3. Cross-checking against the hand-written NumPy backward pass

Reusing the exact same forward/backward math from `16_backpropagation`, on the SAME data, to
confirm PyTorch's autograd and the hand-written NumPy backward pass agree to floating-point
precision.

In [4]:
def relu_backward(dA, Z):
    return dA * (Z > 0)

def softmax(z):
    z_shifted = z - np.max(z, axis=-1, keepdims=True)
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)

# NumPy forward pass (b1=b2=0, matching the zero-initialized biases above)
Z1_manual = X_np @ W1_np
A1_manual = np.maximum(0, Z1_manual)
Z2_manual = A1_manual @ W2_np
probs_manual = softmax(Z2_manual)

n = len(y_np)
one_hot = np.zeros_like(probs_manual)
one_hot[np.arange(n), y_np] = 1.0
dZ2_manual = (probs_manual - one_hot) / n
dW2_manual = A1_manual.T @ dZ2_manual
dA1_manual = dZ2_manual @ W2_np.T
dZ1_manual = relu_backward(dA1_manual, Z1_manual)
dW1_manual = X_np.T @ dZ1_manual

print("max abs difference, dW1 (autograd vs manual):", np.abs(W1.grad.numpy() - dW1_manual).max())
print("max abs difference, dW2 (autograd vs manual):", np.abs(W2.grad.numpy() - dW2_manual).max())

assert np.allclose(W1.grad.numpy(), dW1_manual, atol=1e-4)
assert np.allclose(W2.grad.numpy(), dW2_manual, atol=1e-4)
print("\nPyTorch autograd and the from-scratch NumPy backward pass agree.")


max abs difference, dW1 (autograd vs manual): 8.494747134879521e-09
max abs difference, dW2 (autograd vs manual): 8.659426543877746e-09

PyTorch autograd and the from-scratch NumPy backward pass agree.


## 4. The `.grad` accumulation gotcha

⚠️ **Common Mistake:** calling `.backward()` multiple times without zeroing `.grad` in between
-- gradients ACCUMULATE (sum), they don't get overwritten. This is one of the most common
silent PyTorch training bugs.

In [5]:
w = torch.tensor(2.0, requires_grad=True)

# First backward call.
loss1 = (w * 3) ** 2
loss1.backward()
grad_after_first = w.grad.item()
print("grad after 1st .backward():", grad_after_first)

# Second backward call, WITHOUT zeroing grad first.
loss2 = (w * 3) ** 2
loss2.backward()
grad_after_second = w.grad.item()
print("grad after 2nd .backward() (no zero_grad!):", grad_after_second, "-- DOUBLED, not overwritten!")

assert np.isclose(grad_after_second, 2 * grad_after_first)

# The fix: zero the gradient before each new backward pass.
w.grad.zero_()
loss3 = (w * 3) ** 2
loss3.backward()
grad_after_zeroing = w.grad.item()
print("\ngrad after zero_() + 3rd .backward():", grad_after_zeroing, "-- back to the correct single-step value")
assert np.isclose(grad_after_zeroing, grad_after_first)


grad after 1st .backward(): 36.0
grad after 2nd .backward() (no zero_grad!): 72.0 -- DOUBLED, not overwritten!

grad after zero_() + 3rd .backward(): 36.0 -- back to the correct single-step value


## 5. `torch.no_grad()` for inference

💡 **Intuition:** during evaluation/inference you never call `.backward()`, so tracking the
graph is pure overhead -- `torch.no_grad()` disables it.

In [6]:
with torch.no_grad():
    Z1_eval = X @ W1 + b1
    A1_eval = torch.relu(Z1_eval)
    Z2_eval = A1_eval @ W2 + b2

print("Z2_eval.requires_grad:", Z2_eval.requires_grad, "(False -- no graph was built)")
assert Z2_eval.requires_grad is False

# Outside no_grad, the same computation DOES require grad (since W1/W2 do).
Z2_training_mode = X @ W1 + b1
print("same computation outside no_grad -- requires_grad:", Z2_training_mode.requires_grad)
assert Z2_training_mode.requires_grad is True


Z2_eval.requires_grad: False (False -- no graph was built)
same computation outside no_grad -- requires_grad: True


## 6. Connection to what's next

`20_pytorch_models` wraps exactly this forward-pass logic in `nn.Module`/`nn.Linear`, so you
stop manually tracking `W1`/`b1`/`W2`/`b2` as separate tensors -- the module handles parameter
registration, and `.backward()` + an optimizer (`22_pytorch_training`) handles the rest.

## 7. Interview questions
1. Why does `.grad` accumulate rather than overwrite by default?
2. Walk through why autograd's gradients for the 2-layer MLP match the hand-derived NumPy gradients from `16_backpropagation`.
3. What's the practical difference between `.detach()` and `torch.no_grad()`?
4. Why would skipping `torch.no_grad()` during inference waste memory even though you never call `.backward()`?
5. What error would calling `.backward()` twice on the same graph (without `retain_graph=True`) produce, and why?

## 8. Exercises
1. Build a 3-layer MLP in PyTorch and verify `.backward()`'s gradients against a hand-extended version of `16_backpropagation`'s NumPy backward pass.
2. Intentionally forget to zero gradients across 3 training steps and plot how the (incorrect) accumulated gradient magnitude grows.
3. Measure memory usage (`torch.cuda.memory_allocated()` if on CUDA, or just observe timing on CPU/MPS) with and without `torch.no_grad()` during a forward pass.

## 9. Summary

✅ Autograd automates exactly the chain-rule bookkeeping from `16_backpropagation` — verified
here to match a hand-written backward pass to floating-point precision. `.grad` accumulates
(zero it between steps); `torch.no_grad()` disables tracking for inference. Next:
`20_pytorch_models`.